# Pha S — Buoc 4: Kiem tra do nhay tham so (sensitivity analysis)

**Muc tieu:** xac nhan ket luan chinh (KSG: TE(ho hap->tim) > TE(tim->ho hap), xem notebook 03) KHONG phu thuoc vao 1 lua chon tham so may man - chuan bi cho yeu cau cua Q1/Q2 (reviewer hay hoi 'ket qua co on dinh khi doi tham so khong').

Kiem tra 2 truc:
1. **Dai bang thong loc** (`bandpass_hz`): mac dinh (0.1, 0.5) - thu them (0.15, 0.4) hep hon va (0.05, 0.6) rong hon.
2. **Do dai cua so** (`window_seconds`): mac dinh 30s - thu them 20s va 45s.

Voi MOI to hop, chay lai TOAN BO pipeline (RR+RESP, dong bo, kiem tra dong bo, cat cua so) tren ca 40 ban ghi Fantasia, roi kiem dinh CHINH THUC theo don vi ban ghi (`run_sanity_check_per_record` - xem notebook 03 ve ly do khong dung cua so lam don vi mau). Chi dung KSG (Amortized da biet FAIL tren du lieu that - xem docs/PHASE_S_REPORT.md muc 4, lap lai o day khong co gia tri them).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os
os.environ.setdefault('JAVA_HOME', r'C:\Program Files\Java\jdk-22')
import itertools, yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt, wfdb
from pathlib import Path

from pqrst.data.real.cardiac import rr_from_beat_annotations, interpolate_rr
from pqrst.data.real.respiration import preprocess_respiration, bandpass_filter
from pqrst.data.real.sync import align_to_common_grid, sync_and_window, verify_synchronization
from pqrst.data.real.download import list_local_records
from pqrst.evaluation.sanity_check import bidirectional_te, run_sanity_check_per_record
from pqrst.baselines.ksg import KSGTEEstimator

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cfg = yaml.safe_load(open(BASE/'configs'/'real'/'preprocessing.yaml', encoding='utf-8'))
est_ksg = KSGTEEstimator()
all_records = sorted(list_local_records(BASE / 'data' / 'raw' / 'fantasia', 'fantasia'))
print(f'{len(all_records)} ban ghi Fantasia tren dia.')

40 ban ghi Fantasia tren dia.


## 1. Ham chay toan bo pipeline cho 1 to hop tham so

Tra ve 1 dict cho MOI ban ghi: PASS/FAIL kiem tra dong bo + TE(resp->tim)/TE(tim->resp) trung binh (KSG) neu PASS. Giong logic `process_one_record` o notebook 02c nhung KHONG luu file (chi dung tam thoi cho phan tich do nhay).

In [2]:
def run_one_config(record_id: str, bandpass_hz: tuple, window_seconds: float) -> dict:
    report = {'record_id': record_id}
    try:
        ann = wfdb.rdann(str(BASE / 'data' / 'raw' / 'fantasia' / record_id), 'ecg')
        beat_times_raw = ann.sample / 250.0
        is_normal = np.array([s == 'N' for s in ann.symbol])
        beat_times = beat_times_raw[is_normal]

        t_rr, rr = rr_from_beat_annotations(
            beat_times, exclude_ectopic=True,
            threshold_ratio=cfg['cardiac']['ectopic_threshold_ratio'])
        removed_frac = 1 - len(rr) / max(len(beat_times) - 1, 1)
        if removed_frac > cfg['cardiac']['max_removed_fraction']:
            report['verdict'] = 'FAIL'
            return report

        t_grid_rr, rr_grid = interpolate_rr(t_rr, rr, grid_fs=cfg['grid_fs'])
        rr_grid_filtered = bandpass_filter(rr_grid, cfg['grid_fs'], bandpass_hz)

        record = wfdb.rdrecord(str(BASE / 'data' / 'raw' / 'fantasia' / record_id))
        resp_sig = record.p_signal[:, record.sig_name.index('RESP')]
        t_resp, resp_grid = preprocess_respiration(
            resp_sig, fs=record.fs, grid_fs=cfg['grid_fs'], bandpass=bandpass_hz)

        rr_aligned, resp_aligned = align_to_common_grid(
            t_grid_rr, rr_grid_filtered, t_resp, resp_grid, grid_fs=cfg['grid_fs'])

        sync_check = verify_synchronization(
            rr_aligned, resp_aligned, grid_fs=cfg['grid_fs'], estimator=est_ksg,
            shift_seconds=cfg['sync_check']['shift_seconds'],
            max_ratio_for_pass=cfg['sync_check']['max_ratio_for_pass'])
        if sync_check.get('inconclusive') or not sync_check['passed']:
            report['verdict'] = 'FAIL'
            return report

        w_rr_to_resp = sync_and_window(rr_aligned, resp_aligned, grid_fs=cfg['grid_fs'],
                                       window_seconds=window_seconds, record_id=record_id)
        w_resp_to_rr = sync_and_window(resp_aligned, rr_aligned, grid_fs=cfg['grid_fs'],
                                       window_seconds=window_seconds, record_id=record_id)
        if len(w_rr_to_resp) < 3:  # qua it cua so de tin duoc trung binh
            report['verdict'] = 'FAIL'
            return report

        te_res = bidirectional_te(w_resp_to_rr, w_rr_to_resp, est_ksg)
        report['verdict'] = 'PASS'
        report['te_resp_to_rr'] = te_res['te_forward_mean']
        report['te_rr_to_resp'] = te_res['te_backward_mean']
        report['n_windows'] = te_res['n_windows']
        return report
    except Exception as e:
        report['verdict'] = 'ERROR'
        report['reason'] = f'{type(e).__name__}: {e}'
        return report

## 2. Chay luoi to hop (dai bang thong x do dai cua so)

3 dai bang thong x 3 do dai cua so = 9 to hop (gom 1 trung voi cau hinh mac dinh cua notebook 02c/03 - dung de doi chieu). Chay tren CA 40 ban ghi cho MOI to hop - co the mat vai phut.

In [3]:
bandpass_options = [(0.15, 0.4), (0.1, 0.5), (0.05, 0.6)]  # hep -> mac dinh -> rong
window_options = [20.0, 30.0, 45.0]  # mac dinh = 30.0

grid_summary = []
for bandpass_hz, window_seconds in itertools.product(bandpass_options, window_options):
    reports = [run_one_config(rid, bandpass_hz, window_seconds) for rid in all_records]
    passed = [r for r in reports if r['verdict'] == 'PASS']
    n_pass = len(passed)
    label = f'bandpass={bandpass_hz}, window={window_seconds}s'
    if n_pass < 5:
        print(f'[{label}] CHI {n_pass} ban ghi PASS - qua it de kiem dinh, bo qua.')
        grid_summary.append({'bandpass_hz': str(bandpass_hz), 'window_seconds': window_seconds,
                              'n_pass': n_pass, 'passed': None})
        continue

    fwd = [r['te_resp_to_rr'] for r in passed]
    bwd = [r['te_rr_to_resp'] for r in passed]
    res = run_sanity_check_per_record(fwd, bwd, n_bootstrap=cfg['n_bootstrap'], seed=cfg['seed'])
    print(f"[{label}] N={n_pass}, hieu={res['difference']:.4f}, "
          f"CI=[{res['ci_difference'][0]:.4f},{res['ci_difference'][1]:.4f}], "
          f"Wilcoxon p={res['wilcoxon_p']:.4f}, PASSED={res['passed']}")
    grid_summary.append({
        'bandpass_hz': str(bandpass_hz), 'window_seconds': window_seconds,
        'n_pass': n_pass, 'te_diff': res['difference'],
        'ci_low': res['ci_difference'][0], 'ci_high': res['ci_difference'][1],
        'wilcoxon_p': res['wilcoxon_p'], 'passed': res['passed'],
    })

Sanity check (theo ban ghi, N=15) PASSED: TE forward > TE backward ro ret, CI hieu so khong chua 0, Wilcoxon p=0.0062.
[bandpass=(0.15, 0.4), window=20.0s] N=15, hieu=0.0313, CI=[0.0138,0.0491], Wilcoxon p=0.0062, PASSED=True
Sanity check (theo ban ghi, N=15) PASSED: TE forward > TE backward ro ret, CI hieu so khong chua 0, Wilcoxon p=0.0319.
[bandpass=(0.15, 0.4), window=30.0s] N=15, hieu=0.0258, CI=[0.0037,0.0496], Wilcoxon p=0.0319, PASSED=True
Sanity check (theo ban ghi, N=15) FAILED.
[bandpass=(0.15, 0.4), window=45.0s] N=15, hieu=0.0220, CI=[-0.0037,0.0491], Wilcoxon p=0.0206, PASSED=False
Sanity check (theo ban ghi, N=17) PASSED: TE forward > TE backward ro ret, CI hieu so khong chua 0, Wilcoxon p=0.0008.
[bandpass=(0.1, 0.5), window=20.0s] N=17, hieu=0.0220, CI=[0.0111,0.0347], Wilcoxon p=0.0008, PASSED=True
Sanity check (theo ban ghi, N=17) PASSED: TE forward > TE backward ro ret, CI hieu so khong chua 0, Wilcoxon p=0.0116.
[bandpass=(0.1, 0.5), window=30.0s] N=17, hieu=0.0180

## 3. Bang tong hop + ket luan

In [4]:
summary_df = pd.DataFrame(grid_summary)
out_path = BASE / 'results' / 'phase_s_sensitivity_analysis.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_path, index=False)
print(f'Da luu: {out_path}')
summary_df

Da luu: d:\SPARC Lab\PQRST\results\phase_s_sensitivity_analysis.csv


,bandpass_hz,window_seconds,n_pass,te_diff,ci_low,ci_high,wilcoxon_p,passed
0,"(0.15, 0.4)",20.0,15,0.031340,0.013814,0.049134,0.006226,True
1,"(0.15, 0.4)",30.0,15,0.025844,0.003667,0.049634,0.031860,True
2,"(0.15, 0.4)",45.0,15,0.022031,-0.003672,0.049102,0.020630,False
3,"(0.1, 0.5)",20.0,17,0.021967,0.011106,0.034748,0.000839,True
4,"(0.1, 0.5)",30.0,17,0.018011,0.003947,0.035330,0.011612,True
5,"(0.1, 0.5)",45.0,17,0.015932,-0.001209,0.035251,0.015259,False
6,"(0.05, 0.6)",20.0,19,0.010792,0.001012,0.023597,0.018034,True
7,"(0.05, 0.6)",30.0,19,0.006967,-0.006760,0.024860,0.146749,False
8,"(0.05, 0.6)",45.0,19,0.005487,-0.011597,0.027494,0.270609,False


## 4. KET LUAN DO NHAY

**SUA (review sau khi chay - ket luan nhi phan "tat ca phai PASS" qua khac nghiet):**
day khong phai tieu chi dung de danh gia do nhay. Can tach 2 cau hoi rieng:

1. **Chieu (dau cua hieu so) co on dinh khong?** - day la cau hoi quan trong nhat,
   vi day la chinh tuyen bo khoa hoc (RSA: ho hap->tim).
2. **Y nghia thong ke (CI loai tru 0) co dat o MOI to hop khong?** - cau hoi nay
   PHU THUOC vao suc manh thong ke, tu nhien se yeu di khi cua so dai hon (it cua
   so doc lap hon/ban ghi) hoac bang thong rong hon (pha loang tin hieu ghep noi
   cu the bang noi dung ngoai dai quan tam) - **KHONG dat o moi to hop la ky vong
   HOP LY, khong phai dau hieu ket qua yeu/gia**.

Nhin bang o Muc 3: hieu so **giam dan** khi cua so dai hon (20s->30s->45s, ro rang o
ca 3 dai bang thong) va khi bang thong rong hon (0.15-0.4 -> 0.1-0.5 -> 0.05-0.6) -
day la MOT PATTERN CO THE GIAI THICH (khong ngau nhien): cua so dai hon va bang
thong rong hon deu lam loang tin hieu RSA cu the bang cach dua vao nhieu noi dung
khong lien quan hon.

In [ ]:
valid = summary_df.dropna(subset=['passed'])
n_stable = (valid['passed'] == True).sum()
sign_consistent = (valid['te_diff'] > 0).all()
print(f'Chieu nhat quan (hieu so LUON duong, dung huong RSA) tren {len(valid)}/{len(valid)} to hop: {sign_consistent}')
print(f'Y nghia thong ke (CI loai tru 0) dat o {n_stable}/{len(valid)} to hop')

# Kiem tra pattern giam dan theo do dai cua so / do rong bang thong (giai thich duoc,
# khong phai ngau nhien) - neu dung, cang cung co ket luan la hieu ung THAT (yeu di
# theo huong ky vong khi pha loang tin hieu) chu khong phai nhieu ngau nhien.
by_window = valid.groupby('window_seconds')['te_diff'].mean()
monotonic_window = by_window.is_monotonic_decreasing
print(f'\\nHieu so trung binh theo do dai cua so:\\n{by_window}')
print(f'Giam dan theo do dai cua so (nhu ky vong - cua so dai pha loang tin hieu): {monotonic_window}')

print()
if sign_consistent and monotonic_window:
    print('=== KET LUAN: Chieu ket qua ON DINH TUYET DOI (9/9 to hop). Y nghia thong ke '
          f'dat {n_stable}/{len(valid)} to hop, giam dan CO GIAI THICH DUOC theo cua so '
          'dai hon/bang thong rong hon (pha loang tin hieu) - day la hanh vi HOP LY cua '
          'mot hieu ung THAT co kich thuoc vua phai, khong phai dau hieu ket qua gia/nguu nhien. ===')
else:
    print('=== KET LUAN: Can xem lai - chieu khong nhat quan hoac pattern khong giai thich duoc. ===')